# Reddit Thread Temperature Analysis

Temperature measures engagement intensity, emotional heat, and controversial friction over time using:
- **Velocity (V)**: Activity density
- **Urgency (U)**: Reply speed
- **Emotional Heat (H)**: Linguistic intensity (toxicity + social triggers)
- **Friction (F)**: Controversy based on voting patterns

## 1. Setup and Data Loading

In [1]:
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta
from collections import defaultdict
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from transformers import pipeline
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Load data
DATA_PATH = '/content/drive/MyDrive/amc/data_dump202601160119.csv'
df = pd.read_csv(DATA_PATH)

print(f"Loaded {len(df):,} activities")
print(f"\nActivity types: {df['activity_type'].value_counts().to_dict()}")
print(f"\nDate range: {df['timestamp'].min()} to {df['timestamp'].max()}")
df.head()

Loaded 181,910 activities

Activity types: {'comment': 181448, 'post': 462}

Date range: 2009-06-25T21:47:51+00:00 to 2026-01-16T00:44:07+00:00


,activity_id,activity_type,timestamp,subreddit,author,parent_id,parent_type,content,permalink,score,upvotes,downvotes,upvote_ratio,num_comments,edited
0,t3_g2edod,post,2020-04-16T12:55:55+00:00,news,Billy_Lo,NaN,NaN,"Elon Musk's promised ventilators never delivered to California hospitals, governor's office says...",https://www.reddit.com/r/news/comments/g2edod/elon_musks_promised_ventilators_never_delivered/,79,79,0.0,0.51,380.0,False
1,t1_fnky2o0,comment,2020-04-16T13:14:01+00:00,news,NaN,t3_g2edod,post,[deleted],https://www.reddit.com/r/news/comments/g2edod/elon_musks_promised_ventilators_never_delivered/fn...,779,779,NaN,NaN,NaN,False
2,t1_fnl7yjf,comment,2020-04-16T14:52:05+00:00,news,SomDonkus,t3_g2edod,post,I'm happy that all the top comments can smell this article and op's bullshit a mile away. Why li...,https://www.reddit.com/r/news/comments/g2edod/elon_musks_promised_ventilators_never_delivered/fn...,308,308,NaN,NaN,NaN,False
3,t1_fnl06m4,comment,2020-04-16T13:37:17+00:00,news,Douglaston_prop,t3_g2edod,post,Last I heard California had a surplus and they were sending ventilators out of state where they ...,https://www.reddit.com/r/news/comments/g2edod/elon_musks_promised_ventilators_never_delivered/fn...,62,62,NaN,NaN,NaN,False
4,t1_fnl3nn6,comment,2020-04-16T14:12:13+00:00,news,micahamey,t3_g2edod,post,"Yeah, sure except not only did people receive them earlier this month, but there are pictures as...",https://www.reddit.com/r/news/comments/g2edod/elon_musks_promised_ventilators_never_delivered/fn...,107,107,NaN,NaN,NaN,False


In [4]:
# Parse timestamps
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)

# Ensure content is string
df['content'] = df['content'].fillna('').astype(str)

# Parse numeric columns
df['score'] = pd.to_numeric(df['score'], errors='coerce').fillna(0).astype(int)
df['upvotes'] = pd.to_numeric(df['upvotes'], errors='coerce').fillna(0).astype(int)
df['downvotes'] = pd.to_numeric(df['downvotes'], errors='coerce').fillna(0).astype(int)
df['upvote_ratio'] = pd.to_numeric(df['upvote_ratio'], errors='coerce').fillna(0.5)
df['num_comments'] = pd.to_numeric(df['num_comments'], errors='coerce').fillna(0).astype(int)

print("Data types after parsing:")
print(df.dtypes)

Data types after parsing:
activity_id                   object
activity_type                 object
timestamp        datetime64[ns, UTC]
subreddit                     object
author                        object
parent_id                     object
parent_type                   object
content                       object
permalink                     object
score                          int64
upvotes                        int64
downvotes                      int64
upvote_ratio                 float64
num_comments                   int64
edited                          bool
dtype: object


## 2. Thread Grouping and Time Windowing

In [5]:
def get_root_post_id(activity_id, parent_id, parent_type, activity_type):
    """
    Get the root post ID for an activity.
    - For posts: the activity itself is the root
    - For comments: traverse up to find the root post
    """
    if activity_type == 'post':
        return activity_id
    elif parent_type == 'post':
        return parent_id
    else:
        # For comments replying to comments, we need to trace back
        return parent_id  # Will be resolved in batch processing

In [6]:
# Create a mapping from activity_id to root_post_id
# First pass: identify all posts
posts_df = df[df['activity_type'] == 'post'].copy()
post_ids = set(posts_df['activity_id'].unique())

print(f"Found {len(post_ids):,} unique posts")

# Create parent lookup
parent_lookup = df.set_index('activity_id')[['parent_id', 'parent_type']].to_dict('index')

def find_root_post(activity_id, max_depth=50):
    """Recursively find the root post for any activity."""
    current_id = activity_id
    depth = 0

    while depth < max_depth:
        if current_id in post_ids:
            return current_id

        if current_id not in parent_lookup:
            return None  # Orphaned comment

        parent_info = parent_lookup[current_id]
        parent_id = parent_info['parent_id']

        if parent_id == 'N/A' or pd.isna(parent_id):
            return current_id if current_id in post_ids else None

        current_id = parent_id
        depth += 1

    return None  # Max depth reached

Found 462 unique posts


In [7]:
# Assign root_post_id to each activity
print("Assigning root post IDs...")
root_post_ids = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Finding root posts"):
    if row['activity_type'] == 'post':
        root_post_ids.append(row['activity_id'])
    else:
        root_id = find_root_post(row['activity_id'])
        root_post_ids.append(root_id)

df['root_post_id'] = root_post_ids

# Remove orphaned comments
orphaned = df['root_post_id'].isna().sum()
print(f"Orphaned activities (no root post found): {orphaned:,}")
df = df[df['root_post_id'].notna()].copy()

print(f"\nActivities after cleanup: {len(df):,}")
print(f"Unique threads: {df['root_post_id'].nunique():,}")

Assigning root post IDs...


Finding root posts:   0%|          | 0/181910 [00:00<?, ?it/s]

Orphaned activities (no root post found): 0

Activities after cleanup: 181,910
Unique threads: 462


In [8]:
# Time windowing configuration
WINDOW_SIZE_HOURS = 24

def create_time_windows(thread_df, window_hours=1):
    """
    Create time windows for a thread.
    Returns a list of (window_start, window_end, activities_in_window) tuples.
    """
    if len(thread_df) == 0:
        return []

    # Sort by timestamp
    thread_df = thread_df.sort_values('timestamp')

    # Find thread start time (earliest activity)
    start_time = thread_df['timestamp'].min()
    end_time = thread_df['timestamp'].max()

    # Create windows
    windows = []
    current_start = start_time

    while current_start <= end_time:
        current_end = current_start + timedelta(hours=window_hours)

        # Get activities in this window
        window_mask = (thread_df['timestamp'] >= current_start) & (thread_df['timestamp'] < current_end)
        window_activities = thread_df[window_mask]

        if len(window_activities) > 0:
            windows.append({
                'window_start': current_start,
                'window_end': current_end,
                'activities': window_activities
            })

        current_start = current_end

    return windows

## 3. Temperature Component Calculations

### 3.1 Velocity (V) - Activity Density

In [9]:
def calculate_velocity(window_activities):
    """
    Calculate Velocity V_w = ln(1 + N_comments)

    Measures the volume of conversation using logarithmic scale.
    """
    n_comments = len(window_activities[window_activities['activity_type'] == 'comment'])
    V = np.log(1 + n_comments)
    return V, n_comments

### 3.2 Urgency (U) - Reply Speed

In [10]:
def calculate_urgency(window_activities, all_activities_lookup):
    """
    Calculate Urgency U_w = 1 / ln(e + avg_lag)

    Measures how quickly people are replying.
    """
    comments = window_activities[window_activities['activity_type'] == 'comment']

    if len(comments) == 0:
        return 0.5, None  # Default neutral value

    lags = []
    for _, comment in comments.iterrows():
        parent_id = comment['parent_id']

        if parent_id in all_activities_lookup:
            parent_time = all_activities_lookup[parent_id]
            lag_seconds = (comment['timestamp'] - parent_time).total_seconds()

            # Only consider positive lags (comment after parent)
            if lag_seconds >= 0:
                lags.append(lag_seconds)

    if len(lags) == 0:
        return 0.5, None

    avg_lag = np.mean(lags)
    U = 1 / np.log(np.e + avg_lag)

    return U, avg_lag

### 3.3 Emotional Heat (H) - Linguistic Intensity

In [11]:
# Initialize toxicity model
print("Loading toxic-bert model...")
toxicity_classifier = pipeline(
    "text-classification",
    model="unitary/toxic-bert",
    top_k=None,
    truncation=True,
    max_length=512
)
print("Model loaded!")

Loading toxic-bert model...


config.json:   0%|          | 0.00/811 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda:0


Model loaded!


In [12]:
# Social trigger patterns
PATTERN_YOU = re.compile(r'\b(you|your|yours|u|ur)\b', re.IGNORECASE)
PATTERN_THEM = re.compile(r'\b(they|them|their|theirs|those people)\b', re.IGNORECASE)
PATTERN_ABS = re.compile(r'\b(always|never|everyone|nobody|totally|absolutely|fact|proven)\b', re.IGNORECASE)

def calculate_trigger_density(text):
    """
    Calculate social trigger densities.
    D_pattern = (count / total_words) * 100
    """
    words = text.split()
    total_words = len(words)

    if total_words == 0:
        return 0, 0, 0, 0

    count_you = len(PATTERN_YOU.findall(text))
    count_them = len(PATTERN_THEM.findall(text))
    count_abs = len(PATTERN_ABS.findall(text))

    D_you = (count_you / total_words) * 100
    D_them = (count_them / total_words) * 100
    D_abs = (count_abs / total_words) * 100

    # Weighted sum
    sigma_triggers = (0.2 * D_you) + (0.15 * D_them) + (0.1 * D_abs)

    return sigma_triggers, D_you, D_them, D_abs

In [13]:
def get_toxicity_score(texts, batch_size=32):
    """
    Get average toxicity score for a list of texts using toxic-bert.
    Returns average probability across all toxicity types.
    """
    if not texts:
        return 0.0, {}

    # Filter empty texts and truncate long ones
    valid_texts = [t[:512] for t in texts if t.strip()]

    if not valid_texts:
        return 0.0, {}

    try:
        results = toxicity_classifier(valid_texts, batch_size=batch_size)

        # Collect all toxicity scores
        all_scores = defaultdict(list)

        for result in results:
            for label_score in result:
                label = label_score['label']
                score = label_score['score']

                # toxic-bert returns 'toxic' label with score
                if 'toxic' in label.lower():
                    all_scores['toxic'].append(score)
                all_scores[label].append(score)

        # Calculate average toxic score
        if 'toxic' in all_scores:
            avg_toxic = np.mean(all_scores['toxic'])
        else:
            # Fallback: average all scores
            all_vals = [s for scores in all_scores.values() for s in scores]
            avg_toxic = np.mean(all_vals) if all_vals else 0.0

        return avg_toxic, {k: np.mean(v) for k, v in all_scores.items()}

    except Exception as e:
        print(f"Toxicity error: {e}")
        return 0.0, {}

In [14]:
def calculate_emotional_heat(window_activities, use_toxicity=True):
    """
    Calculate Emotional Heat H_w = (1 + S_toxic) * (1 + sigma_triggers)
    """
    # Combine all text in window
    texts = window_activities['content'].tolist()
    combined_text = ' '.join(texts)

    # Calculate trigger density
    sigma_triggers, D_you, D_them, D_abs = calculate_trigger_density(combined_text)

    # Calculate toxicity
    if use_toxicity and texts:
        avg_toxic, toxicity_breakdown = get_toxicity_score(texts)
    else:
        avg_toxic = 0.0
        toxicity_breakdown = {}

    # Combine into Heat score
    H = (1 + avg_toxic) * (1 + sigma_triggers)

    return H, {
        'toxicity': avg_toxic,
        'toxicity_breakdown': toxicity_breakdown,
        'sigma_triggers': sigma_triggers,
        'D_you': D_you,
        'D_them': D_them,
        'D_abs': D_abs
    }

### 3.4 Friction (F) - Controversy Multiplier

In [15]:
def calculate_friction(window_activities, is_first_window):
    """
    Calculate Friction F_w = 1 + (1 - |2 * (R_up - 0.5)|)

    Only applies to windows containing the original post.
    F = 2.0 when ratio is 0.5 (50% upvotes = maximum controversy)
    F = 1.0 when ratio is 0.0 or 1.0 (no added friction)
    """
    if not is_first_window:
        return 1.0, None

    # Get post's upvote ratio
    posts = window_activities[window_activities['activity_type'] == 'post']

    if len(posts) == 0:
        return 1.0, None

    upvote_ratio = posts.iloc[0]['upvote_ratio']

    # Handle edge cases
    if pd.isna(upvote_ratio):
        upvote_ratio = 0.5

    # Clamp ratio to [0, 1]
    upvote_ratio = max(0.0, min(1.0, upvote_ratio))

    F = 1 + (1 - abs(2 * (upvote_ratio - 0.5)))

    return F, upvote_ratio

### 3.5 Temperature Normalization

In [16]:
def normalize_temperature(raw_energy, M=10.0, k=0.2):
    """
    Normalize raw energy to 0-10 scale using sigmoid.

    T_final = 10 / (1 + exp(-k * (E - M)))

    Parameters:
    - M: Midpoint (energy level considered "active")
    - k: Steepness of the sigmoid curve
    """
    T = 10 / (1 + np.exp(-k * (raw_energy - M)))
    return T

## 4. Full Temperature Calculation Pipeline

In [17]:
def calculate_window_temperature(window_activities, all_activities_lookup, is_first_window, use_toxicity=True):
    """
    Calculate temperature for a single time window.

    E_w = V_w * U_w * H_w * F_w
    T_final = sigmoid_normalize(E_w)
    """
    if len(window_activities) == 0:
        return {
            'temperature': 0,
            'raw_energy': 0,
            'velocity': 0,
            'urgency': 0.5,
            'heat': 1,
            'friction': 1,
            'details': {}
        }

    # Calculate components
    V, n_comments = calculate_velocity(window_activities)
    U, avg_lag = calculate_urgency(window_activities, all_activities_lookup)
    H, heat_details = calculate_emotional_heat(window_activities, use_toxicity)
    F, upvote_ratio = calculate_friction(window_activities, is_first_window)

    # Raw Energy
    E = V * U * H * F

    # Normalized Temperature
    T = normalize_temperature(E)

    return {
        'temperature': T,
        'raw_energy': E,
        'velocity': V,
        'urgency': U,
        'heat': H,
        'friction': F,
        'details': {
            'n_comments': n_comments,
            'avg_lag_seconds': avg_lag,
            'upvote_ratio': upvote_ratio,
            **heat_details
        }
    }

In [18]:
def process_thread(thread_id, thread_df, all_activities_lookup, use_toxicity=True):
    """
    Process a complete thread and calculate temperature for all time windows.
    """
    # Create time windows
    windows = create_time_windows(thread_df, window_hours=WINDOW_SIZE_HOURS)

    if not windows:
        return []

    results = []
    for i, window in enumerate(windows):
        is_first_window = (i == 0)

        temp_result = calculate_window_temperature(
            window['activities'],
            all_activities_lookup,
            is_first_window,
            use_toxicity
        )

        results.append({
            'thread_id': thread_id,
            'window_start': window['window_start'],
            'window_end': window['window_end'],
            'window_index': i,
            'n_activities': len(window['activities']),
            **temp_result
        })

    return results

In [19]:
# Create timestamp lookup for urgency calculation
print("Creating activity timestamp lookup...")
all_activities_lookup = df.set_index('activity_id')['timestamp'].to_dict()
print(f"Lookup created with {len(all_activities_lookup):,} activities")

Creating activity timestamp lookup...
Lookup created with 181,910 activities


In [20]:
# Process all threads
# NOTE: Set use_toxicity=False for faster processing without ML model
USE_TOXICITY = True  # Set to False for faster testing

all_results = []
grouped = df.groupby('root_post_id')

print(f"Processing {len(grouped):,} threads...")

for thread_id, thread_df in tqdm(grouped, desc="Processing threads"):
    thread_results = process_thread(
        thread_id,
        thread_df,
        all_activities_lookup,
        use_toxicity=USE_TOXICITY
    )
    all_results.extend(thread_results)

# Convert to DataFrame
results_df = pd.DataFrame(all_results)
print(f"\nGenerated {len(results_df):,} temperature readings across {results_df['thread_id'].nunique():,} threads")

Processing 462 threads...


Processing threads:   0%|          | 0/462 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



Generated 1,885 temperature readings across 462 threads


In [21]:
# Preview results
results_df.head(10)

,thread_id,window_start,window_end,window_index,n_activities,temperature,raw_energy,velocity,urgency,heat,friction,details
0,t3_1089eph,2023-01-10 13:07:43+00:00,2023-01-11 13:07:43+00:00,0,132,1.650874,1.895739,4.882802,0.111236,1.762776,1.98,"{'n_comments': 131, 'avg_lag_seconds': 8018.618320610687, 'upvote_ratio': 0.51, 'toxicity': 0.03..."
1,t3_1089eph,2023-01-11 13:07:43+00:00,2023-01-12 13:07:43+00:00,1,6,1.243630,0.241267,1.945910,0.091595,1.353645,1.00,"{'n_comments': 6, 'avg_lag_seconds': 55138.333333333336, 'upvote_ratio': None, 'toxicity': 0.015..."
2,t3_1089eph,2023-01-15 13:07:43+00:00,2023-01-16 13:07:43+00:00,2,1,1.203500,0.054401,0.693147,0.078423,1.000770,1.00,"{'n_comments': 1, 'avg_lag_seconds': 345012.0, 'upvote_ratio': None, 'toxicity': 0.0007702795119..."
3,t3_1089eph,2023-01-16 13:07:43+00:00,2023-01-17 13:07:43+00:00,3,1,1.203219,0.053074,0.693147,0.075653,1.012123,1.00,"{'n_comments': 1, 'avg_lag_seconds': 550312.0, 'upvote_ratio': None, 'toxicity': 0.0121227688117..."
4,t3_1089eph,2023-01-30 13:07:43+00:00,2023-01-31 13:07:43+00:00,4,1,1.206711,0.069548,0.693147,0.069521,1.443267,1.00,"{'n_comments': 1, 'avg_lag_seconds': 1765935.0, 'upvote_ratio': None, 'toxicity': 0.001450668207..."
5,t3_10eydvz,2023-01-18 04:50:04+00:00,2023-01-19 04:50:04+00:00,0,40,1.506789,1.353567,3.688879,0.109393,1.677133,2.00,"{'n_comments': 39, 'avg_lag_seconds': 9330.97435897436, 'upvote_ratio': 0.5, 'toxicity': 0.05397..."
6,t3_10eydvz,2023-01-22 04:50:04+00:00,2023-01-23 04:50:04+00:00,1,1,1.203578,0.054768,0.693147,0.078372,1.008183,1.00,"{'n_comments': 1, 'avg_lag_seconds': 347919.0, 'upvote_ratio': None, 'toxicity': 0.0081829884438..."
7,t3_117sif,2012-10-09 21:31:57+00:00,2012-10-10 21:31:57+00:00,0,6,1.451133,1.132728,1.791759,0.093933,3.469191,1.94,"{'n_comments': 5, 'avg_lag_seconds': 42019.0, 'upvote_ratio': 0.47, 'toxicity': 0.39087361925380..."
8,t3_11bqd8z,2023-02-25 17:24:37+00:00,2023-02-26 17:24:37+00:00,0,125,1.562728,1.568869,4.828314,0.102871,1.735514,1.82,"{'n_comments': 124, 'avg_lag_seconds': 16660.120967741936, 'upvote_ratio': 0.59, 'toxicity': 0.0..."
9,t3_11bqd8z,2023-02-26 17:24:37+00:00,2023-02-27 17:24:37+00:00,1,15,1.283792,0.423169,2.772589,0.091551,1.667117,1.00,"{'n_comments': 15, 'avg_lag_seconds': 55428.86666666667, 'upvote_ratio': None, 'toxicity': 0.030..."


In [22]:
# Summary statistics
print("Temperature Statistics:")
print(results_df['temperature'].describe())

print("\nRaw Energy Statistics:")
print(results_df['raw_energy'].describe())

Temperature Statistics:
count    1885.000000
mean        1.325503
std         0.172396
min         1.192029
25%         1.212341
50%         1.242950
75%         1.346157
max         2.068409
Name: temperature, dtype: float64

Raw Energy Statistics:
count    1885.000000
mean        0.570541
std         0.695531
min         0.000000
25%         0.096025
50%         0.238141
75%         0.696251
max         3.279632
Name: raw_energy, dtype: float64


## 5. Interactive Visualizations

In [23]:
# Get thread info for labeling
thread_info = posts_df.set_index('activity_id')[['content', 'num_comments', 'score']].to_dict('index')

def get_thread_title(thread_id, max_len=50):
    """Get truncated title for a thread."""
    if thread_id in thread_info:
        content = thread_info[thread_id]['content']
        if len(content) > max_len:
            return content[:max_len] + '...'
        return content
    return thread_id

### 5.1 Temperature Distribution

In [24]:
fig = px.histogram(
    results_df,
    x='temperature',
    nbins=50,
    title='Distribution of Temperature Scores',
    labels={'temperature': 'Temperature (0-10)', 'count': 'Number of Windows'}
)
fig.update_layout(
    xaxis_range=[0, 10],
    bargap=0.1
)
fig.show()

### 5.2 Top Hottest Threads

In [25]:
# Find threads with highest peak temperature
peak_temps = results_df.groupby('thread_id').agg({
    'temperature': 'max',
    'raw_energy': 'max',
    'n_activities': 'sum'
}).reset_index()

peak_temps = peak_temps.sort_values('temperature', ascending=False).head(20)
peak_temps['title'] = peak_temps['thread_id'].apply(lambda x: get_thread_title(x, 60))

fig = px.bar(
    peak_temps,
    x='temperature',
    y='title',
    orientation='h',
    title='Top 20 Hottest Threads (Peak Temperature)',
    labels={'temperature': 'Peak Temperature', 'title': 'Thread'},
    color='temperature',
    color_continuous_scale='RdYlGn_r'
)
fig.update_layout(height=600, yaxis={'categoryorder': 'total ascending'})
fig.show()

### 5.3 Temperature Over Time for Selected Threads

In [26]:
# Select top N threads by peak temperature
TOP_N = 10
top_thread_ids = peak_temps.head(TOP_N)['thread_id'].tolist()

top_threads_df = results_df[results_df['thread_id'].isin(top_thread_ids)].copy()
top_threads_df['title'] = top_threads_df['thread_id'].apply(lambda x: get_thread_title(x, 40))

fig = px.line(
    top_threads_df,
    x='window_index',
    y='temperature',
    color='title',
    title=f'Temperature Over Time - Top {TOP_N} Hottest Threads',
    labels={
        'window_index': 'Time Window (hours from thread start)',
        'temperature': 'Temperature',
        'title': 'Thread'
    },
    markers=True
)
fig.update_layout(height=500)
fig.show()

### 5.4 Component Breakdown

In [27]:
# Component correlation analysis
components = ['velocity', 'urgency', 'heat', 'friction', 'temperature']
corr_matrix = results_df[components].corr()

fig = px.imshow(
    corr_matrix,
    title='Component Correlation Matrix',
    labels=dict(color='Correlation'),
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    text_auto='.2f'
)
fig.show()

In [28]:
# Scatter plot of components
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Velocity vs Temperature', 'Urgency vs Temperature',
                    'Heat vs Temperature', 'Friction vs Temperature')
)

# Sample for performance
sample_df = results_df.sample(min(5000, len(results_df)))

fig.add_trace(
    go.Scatter(x=sample_df['velocity'], y=sample_df['temperature'], mode='markers',
               marker=dict(size=3, opacity=0.5), name='Velocity'),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=sample_df['urgency'], y=sample_df['temperature'], mode='markers',
               marker=dict(size=3, opacity=0.5), name='Urgency'),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=sample_df['heat'], y=sample_df['temperature'], mode='markers',
               marker=dict(size=3, opacity=0.5), name='Heat'),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=sample_df['friction'], y=sample_df['temperature'], mode='markers',
               marker=dict(size=3, opacity=0.5), name='Friction'),
    row=2, col=2
)

fig.update_layout(height=700, title_text='Temperature Components Analysis', showlegend=False)
fig.show()

### 5.5 Temperature Timeline Heatmap

In [29]:
# Create heatmap for top threads
TOP_N_HEATMAP = 15
heatmap_threads = peak_temps.head(TOP_N_HEATMAP)['thread_id'].tolist()

# Pivot data for heatmap
heatmap_df = results_df[results_df['thread_id'].isin(heatmap_threads)].copy()
heatmap_df['title'] = heatmap_df['thread_id'].apply(lambda x: get_thread_title(x, 35))

# Limit window index to first 24 hours
heatmap_df = heatmap_df[heatmap_df['window_index'] < 24]

pivot_df = heatmap_df.pivot_table(
    index='title',
    columns='window_index',
    values='temperature',
    aggfunc='first'
).fillna(0)

fig = px.imshow(
    pivot_df,
    title='Temperature Heatmap - Top Threads (First 24 Hours)',
    labels=dict(x='Hour', y='Thread', color='Temperature'),
    color_continuous_scale='RdYlGn_r',
    aspect='auto'
)
fig.update_layout(height=500)
fig.show()

### 5.6 Detailed Thread Explorer

In [30]:
def explore_thread(thread_id):
    """
    Create detailed visualization for a single thread.
    """
    thread_results = results_df[results_df['thread_id'] == thread_id].copy()

    if len(thread_results) == 0:
        print(f"Thread {thread_id} not found")
        return

    title = get_thread_title(thread_id, 80)

    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=('Temperature Over Time', 'Component Breakdown'),
        shared_xaxes=True,
        vertical_spacing=0.15
    )

    # Temperature line
    fig.add_trace(
        go.Scatter(
            x=thread_results['window_index'],
            y=thread_results['temperature'],
            mode='lines+markers',
            name='Temperature',
            line=dict(color='red', width=3),
            marker=dict(size=8)
        ),
        row=1, col=1
    )

    # Component stacked area
    fig.add_trace(
        go.Scatter(
            x=thread_results['window_index'],
            y=thread_results['velocity'],
            mode='lines',
            name='Velocity',
            stackgroup='components'
        ),
        row=2, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=thread_results['window_index'],
            y=thread_results['urgency'],
            mode='lines',
            name='Urgency',
            stackgroup='components'
        ),
        row=2, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=thread_results['window_index'],
            y=thread_results['heat'],
            mode='lines',
            name='Heat',
            stackgroup='components'
        ),
        row=2, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=thread_results['window_index'],
            y=thread_results['friction'],
            mode='lines',
            name='Friction',
            stackgroup='components'
        ),
        row=2, col=1
    )

    fig.update_layout(
        height=600,
        title_text=f'Thread Analysis: {title}',
        xaxis2_title='Time Window (hours)',
        yaxis_title='Temperature (0-10)',
        yaxis2_title='Component Value'
    )

    fig.show()

    # Print summary stats
    print(f"\nThread Summary:")
    print(f"  Peak Temperature: {thread_results['temperature'].max():.2f}")
    print(f"  Average Temperature: {thread_results['temperature'].mean():.2f}")
    print(f"  Total Windows: {len(thread_results)}")
    print(f"  Total Activities: {thread_results['n_activities'].sum()}")

In [31]:
# Explore the hottest thread
hottest_thread = peak_temps.iloc[0]['thread_id']
explore_thread(hottest_thread)


Thread Summary:
  Peak Temperature: 2.07
  Average Temperature: 1.34
  Total Windows: 10
  Total Activities: 2244


### 5.7 Interactive Thread Selector

In [32]:
# Create dropdown widget for thread selection
from ipywidgets import interact, Dropdown

# Get top 50 threads by temperature
top_50_threads = peak_temps.head(50)
thread_options = {
    f"{get_thread_title(row['thread_id'], 50)} (T={row['temperature']:.1f})": row['thread_id']
    for _, row in top_50_threads.iterrows()
}

@interact(thread=Dropdown(options=thread_options, description='Thread:'))
def interactive_explore(thread):
    explore_thread(thread)

interactive(children=(Dropdown(description='Thread:', options={'Christian Mingle must let LGBT singles use dat…

## 6. Export Results

In [33]:
# Export temperature results to CSV
output_path = 'temperature_results.csv'

# Add thread title for readability
export_df = results_df.copy()
export_df['thread_title'] = export_df['thread_id'].apply(lambda x: get_thread_title(x, 100))

# Reorder columns
cols = ['thread_id', 'thread_title', 'window_start', 'window_end', 'window_index',
        'temperature', 'raw_energy', 'velocity', 'urgency', 'heat', 'friction',
        'n_activities', 'details']
export_df = export_df[[c for c in cols if c in export_df.columns]]

export_df.to_csv(output_path, index=False)
print(f"Results exported to {output_path}")
print(f"Total records: {len(export_df):,}")

Results exported to temperature_results.csv
Total records: 1,885


In [34]:
# Export thread-level summary
summary_path = 'temperature_summary.csv'

thread_summary = results_df.groupby('thread_id').agg({
    'temperature': ['max', 'mean', 'std'],
    'raw_energy': ['max', 'mean'],
    'velocity': 'mean',
    'urgency': 'mean',
    'heat': 'mean',
    'friction': 'max',
    'n_activities': 'sum',
    'window_index': 'max'
}).reset_index()

thread_summary.columns = ['_'.join(col).strip('_') for col in thread_summary.columns]
thread_summary['thread_title'] = thread_summary['thread_id'].apply(lambda x: get_thread_title(x, 100))
thread_summary = thread_summary.sort_values('temperature_max', ascending=False)

thread_summary.to_csv(summary_path, index=False)
print(f"Summary exported to {summary_path}")
print(f"Total threads: {len(thread_summary):,}")

Summary exported to temperature_summary.csv
Total threads: 462


## 7. Quick Analysis Summary

In [35]:
print("=" * 60)
print("TEMPERATURE ANALYSIS SUMMARY")
print("=" * 60)

print(f"\nDataset Overview:")
print(f"  Total activities: {len(df):,}")
print(f"  Total threads: {df['root_post_id'].nunique():,}")
print(f"  Time windows analyzed: {len(results_df):,}")

print(f"\nTemperature Distribution:")
print(f"  Min: {results_df['temperature'].min():.2f}")
print(f"  Max: {results_df['temperature'].max():.2f}")
print(f"  Mean: {results_df['temperature'].mean():.2f}")
print(f"  Median: {results_df['temperature'].median():.2f}")
print(f"  Std Dev: {results_df['temperature'].std():.2f}")

print(f"\nTop 5 Hottest Threads:")
for i, row in peak_temps.head(5).iterrows():
    title = get_thread_title(row['thread_id'], 50)
    print(f"  {row['temperature']:.2f} - {title}")

print(f"\nHigh Temperature Windows (T > 7):")
high_temp = results_df[results_df['temperature'] > 7]
print(f"  Count: {len(high_temp):,} ({100*len(high_temp)/len(results_df):.1f}% of all windows)")

print("\n" + "=" * 60)

TEMPERATURE ANALYSIS SUMMARY

Dataset Overview:
  Total activities: 181,910
  Total threads: 462
  Time windows analyzed: 1,885

Temperature Distribution:
  Min: 1.19
  Max: 2.07
  Mean: 1.33
  Median: 1.24
  Std Dev: 0.17

Top 5 Hottest Threads:
  2.07 - Christian Mingle must let LGBT singles use dating ...
  2.07 - A black social worker called 911 because she was a...
  1.95 - White supremacist allegedly caught on video punchi...
  1.94 - 'Kill the NRA' message appears on billboard on Int...
  1.94 - Restaurant closes after facing backlash for not al...

High Temperature Windows (T > 7):
  Count: 0 (0.0% of all windows)

